# KLA AI Hackathon - Denoising + Super-Resolution Notebook

This notebook is designed for **Kaggle Code Competition** execution. It:
1. Loads paired training data from `train/NoisyLR` and `train/GT`
2. Trains a compact U-Net style restoration model with a **balanced PSNR + SSIM objective**
3. Reports validation PSNR and SSIM per epoch
4. Runs inference for all test images in `test/NoisyLR`
5. Saves outputs as `.npy` files and creates `submission.csv` in the required base64 format

> For Kaggle submission, set `DATA_ROOT = '/kaggle/input/competition-data'`.


In [ ]:
import os
import random
import math
import base64
from io import BytesIO
from pathlib import Path

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

# Reproducibility
def seed_everything(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

seed_everything(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)


In [ ]:
# ============================
# Configuration
# ============================

# Local testing in this repository
DATA_ROOT = Path('.')

# Kaggle competition path (uncomment this on Kaggle)
# DATA_ROOT = Path('/kaggle/input/competition-data')

TRAIN_NOISY_DIR = DATA_ROOT / 'train' / 'NoisyLR'
TRAIN_GT_DIR    = DATA_ROOT / 'train' / 'GT'
TEST_DIR        = DATA_ROOT / 'test' / 'NoisyLR'

WORKING_DIR = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path('.')
SUBMISSION_DIR = WORKING_DIR / 'submission'
SUBMISSION_DIR.mkdir(parents=True, exist_ok=True)

LR_SIZE = 128
IMG_SIZE = 256
SCALE_FACTOR = IMG_SIZE // LR_SIZE

BATCH_SIZE = 16
NUM_EPOCHS = 20
LR = 2e-4
VAL_RATIO = 0.1
NUM_WORKERS = 2

# Loss balance (tune these for PSNR/SSIM tradeoff)
LAMBDA_L1 = 0.6
LAMBDA_SSIM = 0.4

# Optional: for quick iteration, use smaller subset (None = full train)
MAX_TRAIN_SAMPLES = None

print('Train noisy dir exists:', TRAIN_NOISY_DIR.exists())
print('Train GT dir exists   :', TRAIN_GT_DIR.exists())
print('Test dir exists       :', TEST_DIR.exists())
print(f'Super-resolution scale: {SCALE_FACTOR}x ({LR_SIZE} -> {IMG_SIZE})')


In [ ]:
# ============================
# Dataset
# ============================

class NpyPairDataset(Dataset):
    def __init__(self, noisy_files, gt_dir, lr_size=128, hr_size=256, augment=False):
        self.noisy_files = noisy_files
        self.gt_dir = Path(gt_dir)
        self.lr_size = lr_size
        self.hr_size = hr_size
        self.augment = augment

    def __len__(self):
        return len(self.noisy_files)

    @staticmethod
    def _resize_np(img, size):
        ten = torch.from_numpy(img).unsqueeze(0).unsqueeze(0)
        ten = F.interpolate(ten, size=(size, size), mode='bilinear', align_corners=False)
        return ten.squeeze(0).squeeze(0).numpy()

    def _augment(self, noisy, gt):
        # geometric augmentations shared by LR and HR pairs
        if random.random() < 0.5:
            noisy = np.flip(noisy, axis=0).copy()
            gt = np.flip(gt, axis=0).copy()
        if random.random() < 0.5:
            noisy = np.flip(noisy, axis=1).copy()
            gt = np.flip(gt, axis=1).copy()
        k = random.randint(0, 3)
        if k:
            noisy = np.rot90(noisy, k=k).copy()
            gt = np.rot90(gt, k=k).copy()
        return noisy, gt

    def __getitem__(self, idx):
        noisy_path = self.noisy_files[idx]
        gt_path = self.gt_dir / noisy_path.name

        noisy = np.load(noisy_path).astype(np.float32)
        gt = np.load(gt_path).astype(np.float32)

        noisy = np.clip(noisy, 0.0, 1.0)
        gt = np.clip(gt, 0.0, 1.0)

        # Enforce the expected SR training dimensions: 128x128 -> 256x256
        if noisy.shape != (self.lr_size, self.lr_size):
            noisy = self._resize_np(noisy, self.lr_size)
        if gt.shape != (self.hr_size, self.hr_size):
            gt = self._resize_np(gt, self.hr_size)

        if self.augment:
            noisy, gt = self._augment(noisy, gt)

        noisy = torch.from_numpy(noisy).unsqueeze(0)
        gt = torch.from_numpy(gt).unsqueeze(0)
        return noisy, gt


def make_splits(train_noisy_dir, val_ratio=0.1, max_samples=None):
    files = sorted(Path(train_noisy_dir).glob('*.npy'))
    if max_samples is not None:
        files = files[:max_samples]

    random.shuffle(files)
    n_val = max(1, int(len(files) * val_ratio))
    val_files = files[:n_val]
    train_files = files[n_val:]
    return train_files, val_files

train_files, val_files = make_splits(TRAIN_NOISY_DIR, VAL_RATIO, MAX_TRAIN_SAMPLES)
print(f'Train samples: {len(train_files)}, Val samples: {len(val_files)}')

train_ds = NpyPairDataset(train_files, TRAIN_GT_DIR, lr_size=LR_SIZE, hr_size=IMG_SIZE, augment=True)
val_ds = NpyPairDataset(val_files, TRAIN_GT_DIR, lr_size=LR_SIZE, hr_size=IMG_SIZE, augment=False)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=NUM_WORKERS, pin_memory=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False,
                        num_workers=NUM_WORKERS, pin_memory=True)


In [ ]:
# ============================
# Model: Restormer-style Transformer for Denoise + Super-Resolution
# ============================

class BiasFreeLayerNorm(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(dim))

    def forward(self, x):
        # x: [B, C, H, W]
        var = x.var(dim=1, keepdim=True, unbiased=False)
        return x / torch.sqrt(var + 1e-6) * self.weight.view(1, -1, 1, 1)


class WithBiasLayerNorm(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(dim))
        self.bias = nn.Parameter(torch.zeros(dim))

    def forward(self, x):
        mean = x.mean(dim=1, keepdim=True)
        var = x.var(dim=1, keepdim=True, unbiased=False)
        x = (x - mean) / torch.sqrt(var + 1e-6)
        return x * self.weight.view(1, -1, 1, 1) + self.bias.view(1, -1, 1, 1)


class LayerNorm2d(nn.Module):
    def __init__(self, dim, layer_norm_type='WithBias'):
        super().__init__()
        self.body = WithBiasLayerNorm(dim) if layer_norm_type == 'WithBias' else BiasFreeLayerNorm(dim)

    def forward(self, x):
        return self.body(x)


class FeedForward(nn.Module):
    def __init__(self, dim, ffn_expansion_factor=2.66):
        super().__init__()
        hidden = int(dim * ffn_expansion_factor)
        self.project_in = nn.Conv2d(dim, hidden * 2, kernel_size=1)
        self.dwconv = nn.Conv2d(hidden * 2, hidden * 2, kernel_size=3, padding=1, groups=hidden * 2)
        self.project_out = nn.Conv2d(hidden, dim, kernel_size=1)

    def forward(self, x):
        x = self.project_in(x)
        x1, x2 = self.dwconv(x).chunk(2, dim=1)
        x = F.gelu(x1) * x2
        return self.project_out(x)


class Attention(nn.Module):
    def __init__(self, dim, num_heads=4):
        super().__init__()
        self.num_heads = num_heads
        self.temperature = nn.Parameter(torch.ones(num_heads, 1, 1))

        self.qkv = nn.Conv2d(dim, dim * 3, kernel_size=1)
        self.qkv_dwconv = nn.Conv2d(dim * 3, dim * 3, kernel_size=3, padding=1, groups=dim * 3)
        self.project_out = nn.Conv2d(dim, dim, kernel_size=1)

    def forward(self, x):
        b, c, h, w = x.shape
        qkv = self.qkv_dwconv(self.qkv(x))
        q, k, v = qkv.chunk(3, dim=1)

        q = q.reshape(b, self.num_heads, c // self.num_heads, h * w)
        k = k.reshape(b, self.num_heads, c // self.num_heads, h * w)
        v = v.reshape(b, self.num_heads, c // self.num_heads, h * w)

        q = F.normalize(q, dim=2)
        k = F.normalize(k, dim=2)

        attn = (q.transpose(-2, -1) @ k) * self.temperature
        attn = attn.softmax(dim=-1)

        out = v @ attn.transpose(-2, -1)
        out = out.reshape(b, c, h, w)
        return self.project_out(out)


class TransformerBlock(nn.Module):
    def __init__(self, dim, num_heads=4, ffn_expansion_factor=2.66, layer_norm_type='WithBias'):
        super().__init__()
        self.norm1 = LayerNorm2d(dim, layer_norm_type)
        self.attn = Attention(dim, num_heads=num_heads)
        self.norm2 = LayerNorm2d(dim, layer_norm_type)
        self.ffn = FeedForward(dim, ffn_expansion_factor)

    def forward(self, x):
        x = x + self.attn(self.norm1(x))
        x = x + self.ffn(self.norm2(x))
        return x


class RestormerSR(nn.Module):
    def __init__(self, inp_channels=1, out_channels=1, dim=48, num_blocks=6, num_heads=4, scale=2):
        super().__init__()
        self.patch_embed = nn.Conv2d(inp_channels, dim, kernel_size=3, padding=1)
        self.transformer = nn.Sequential(*[
            TransformerBlock(dim=dim, num_heads=num_heads, ffn_expansion_factor=2.66, layer_norm_type='WithBias')
            for _ in range(num_blocks)
        ])
        self.refine = nn.Conv2d(dim, dim, kernel_size=3, padding=1)

        self.upsample = nn.Sequential(
            nn.Conv2d(dim, dim * (scale ** 2), kernel_size=3, padding=1),
            nn.PixelShuffle(scale),
            nn.Conv2d(dim, out_channels, kernel_size=3, padding=1),
        )

    def forward(self, x):
        feat = self.patch_embed(x)
        trans = self.transformer(feat)
        feat = feat + self.refine(trans)
        out = self.upsample(feat)
        return torch.sigmoid(out)


model = RestormerSR(inp_channels=1, out_channels=1, dim=48, num_blocks=6, num_heads=4, scale=SCALE_FACTOR).to(device)
print(f'Model params: {sum(p.numel() for p in model.parameters())/1e6:.2f}M')


In [ ]:
# ============================
# Metrics & loss (PSNR + SSIM balanced optimization)
# ============================

def psnr_torch(pred, target, max_val=1.0, eps=1e-8):
    mse = F.mse_loss(pred, target)
    return 20 * torch.log10(torch.tensor(max_val, device=pred.device)) - 10 * torch.log10(mse + eps)


def _gaussian_window(window_size=11, sigma=1.5, channels=1, device='cpu'):
    coords = torch.arange(window_size, dtype=torch.float32, device=device) - window_size // 2
    g = torch.exp(-(coords ** 2) / (2 * sigma ** 2))
    g = g / g.sum()
    window_1d = g.view(1, 1, 1, -1)
    window_2d = window_1d.transpose(-1, -2) @ window_1d
    window_2d = window_2d / window_2d.sum()
    window = window_2d.expand(channels, 1, window_size, window_size).contiguous()
    return window


def ssim_torch(img1, img2, window_size=11, sigma=1.5, data_range=1.0, eps=1e-8):
    C1 = (0.01 * data_range) ** 2
    C2 = (0.03 * data_range) ** 2

    channels = img1.shape[1]
    window = _gaussian_window(window_size, sigma, channels, img1.device)

    mu1 = F.conv2d(img1, window, padding=window_size // 2, groups=channels)
    mu2 = F.conv2d(img2, window, padding=window_size // 2, groups=channels)

    mu1_sq = mu1 * mu1
    mu2_sq = mu2 * mu2
    mu1_mu2 = mu1 * mu2

    sigma1_sq = F.conv2d(img1 * img1, window, padding=window_size // 2, groups=channels) - mu1_sq
    sigma2_sq = F.conv2d(img2 * img2, window, padding=window_size // 2, groups=channels) - mu2_sq
    sigma12 = F.conv2d(img1 * img2, window, padding=window_size // 2, groups=channels) - mu1_mu2

    ssim_map = ((2 * mu1_mu2 + C1) * (2 * sigma12 + C2)) / ((mu1_sq + mu2_sq + C1) * (sigma1_sq + sigma2_sq + C2) + eps)
    return ssim_map.mean()


def restoration_loss(pred, target, lambda_l1=0.6, lambda_ssim=0.4):
    l1 = F.l1_loss(pred, target)
    ssim_loss = 1.0 - ssim_torch(pred, target)
    return lambda_l1 * l1 + lambda_ssim * ssim_loss, l1.detach(), (1.0 - ssim_loss).detach()


In [ ]:
# ============================
# Training loop
# ============================

optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS)

best_val_score = -1.0
best_path = WORKING_DIR / 'best_model.pt'
history = []

for epoch in range(1, NUM_EPOCHS + 1):
    model.train()
    train_loss = 0.0

    for noisy, gt in train_loader:
        noisy = noisy.to(device, non_blocking=True)
        gt = gt.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)
        pred = model(noisy)

        # Ensure we always optimize 256x256 prediction against 256x256 GT
        if pred.shape[-2:] != gt.shape[-2:]:
            raise RuntimeError(f'Shape mismatch: pred={pred.shape}, gt={gt.shape}')

        loss, _, _ = restoration_loss(pred, gt, LAMBDA_L1, LAMBDA_SSIM)
        loss.backward()
        optimizer.step()

        train_loss += loss.item() * noisy.size(0)

    train_loss /= len(train_loader.dataset)

    model.eval()
    val_loss = 0.0
    val_psnr = 0.0
    val_ssim = 0.0

    with torch.no_grad():
        for noisy, gt in val_loader:
            noisy = noisy.to(device, non_blocking=True)
            gt = gt.to(device, non_blocking=True)

            pred = model(noisy)
            if pred.shape[-2:] != gt.shape[-2:]:
                raise RuntimeError(f'Validation shape mismatch: pred={pred.shape}, gt={gt.shape}')

            loss, _, _ = restoration_loss(pred, gt, LAMBDA_L1, LAMBDA_SSIM)

            val_loss += loss.item() * noisy.size(0)
            val_psnr += psnr_torch(pred, gt).item() * noisy.size(0)
            val_ssim += ssim_torch(pred, gt).item() * noisy.size(0)

    val_loss /= len(val_loader.dataset)
    val_psnr /= len(val_loader.dataset)
    val_ssim /= len(val_loader.dataset)

    # Combined score: prioritize balanced quality
    combined_score = 0.5 * (val_psnr / 40.0) + 0.5 * val_ssim

    if combined_score > best_val_score:
        best_val_score = combined_score
        torch.save(model.state_dict(), best_path)

    scheduler.step()

    history.append({
        'epoch': epoch,
        'train_loss': train_loss,
        'val_loss': val_loss,
        'val_psnr': val_psnr,
        'val_ssim': val_ssim,
        'combined_score': combined_score,
    })

    print(
        f"Epoch {epoch:02d}/{NUM_EPOCHS} | "
        f"train_loss={train_loss:.4f} | val_loss={val_loss:.4f} | "
        f"val_psnr={val_psnr:.3f} | val_ssim={val_ssim:.4f} | score={combined_score:.4f}"
    )

print('Best model saved to:', best_path)


In [ ]:
# Training summary table
hist_df = pd.DataFrame(history)
hist_df.tail(10)


In [ ]:
# ============================
# Inference on test set + required .npy output files
# ============================

# Load best model
model.load_state_dict(torch.load(best_path, map_location=device))
model.eval()

test_files = sorted(TEST_DIR.glob('*.npy'))
print('Test files:', len(test_files))

with torch.no_grad():
    for file in test_files:
        arr = np.load(file).astype(np.float32)
        arr = np.clip(arr, 0.0, 1.0)

        if arr.shape != (LR_SIZE, LR_SIZE):
            ten = torch.from_numpy(arr).unsqueeze(0).unsqueeze(0)
            ten = F.interpolate(ten, size=(LR_SIZE, LR_SIZE), mode='bilinear', align_corners=False)
            arr = ten.squeeze(0).squeeze(0).numpy()

        inp = torch.from_numpy(arr).unsqueeze(0).unsqueeze(0).to(device)
        pred = model(inp).squeeze(0).squeeze(0).cpu().numpy().astype(np.float32)

        # Competition constraints
        pred = np.nan_to_num(pred, nan=0.0, posinf=1.0, neginf=0.0)
        pred = np.clip(pred, 0.0, 1.0).astype(np.float32)

        if pred.shape != (IMG_SIZE, IMG_SIZE):
            raise RuntimeError(f'Inference output shape mismatch for {file.name}: {pred.shape}')

        out_path = SUBMISSION_DIR / file.name
        np.save(out_path, pred)

print('Saved .npy predictions to:', SUBMISSION_DIR)


In [ ]:
# ============================
# Build submission.csv exactly as required by competition
# ============================

rows = []
files = sorted([f for f in os.listdir(SUBMISSION_DIR) if f.endswith('.npy')])

for idx, file in enumerate(files, start=1):
    path = SUBMISSION_DIR / file
    arr = np.load(path)

    # Basic compliance checks
    assert arr.shape == (IMG_SIZE, IMG_SIZE), f"Bad shape in {file}: {arr.shape}"
    assert arr.dtype == np.float32, f"Bad dtype in {file}: {arr.dtype}"
    assert np.isfinite(arr).all(), f"NaN/Inf in {file}"

    buffer = BytesIO()
    np.save(buffer, arr)
    encoded = base64.b64encode(buffer.getvalue()).decode()

    rows.append({'id': idx, 'npy_base64': encoded})

submission_df = pd.DataFrame(rows)
submission_csv_path = WORKING_DIR / 'submission.csv'
submission_df.to_csv(submission_csv_path, index=False)

print('Submission created with', len(submission_df), 'rows')
print('submission.csv path:', submission_csv_path)
submission_df.head()


In [ ]:
# Optional: quick report of final validation best epoch
best_row = hist_df.iloc[hist_df['combined_score'].argmax()]
print('Best epoch:', int(best_row['epoch']))
print('Best val PSNR:', float(best_row['val_psnr']))
print('Best val SSIM:', float(best_row['val_ssim']))
print('Best combined score:', float(best_row['combined_score']))
